# histogram_grid_classic_control

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import itertools

import matplotlib.pyplot as plt
import numpy as np

from analysis.curves import lifetime_average, return_curve, subsample
from analysis.stats import best_by_mean, mean_ci
from main import load_results

from config import EXPERIMENT, LR_SWEEP, NUM_BINS_SWEEP, SIGMA_RATIO_SWEEP, SUPPORT_RANGES

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each environment's stored runs per (support, SIGMA_RATIO, NUM_BINS, LR), if stored.
ENVIRONMENTS = ["cartpole", "mountaincar"]

results = {}
for environment, support in itertools.product(ENVIRONMENTS, SUPPORT_RANGES):
    name = f"dqn_histogram_{environment}_support{support}"
    for sigma, bins, lr in itertools.product(SIGMA_RATIO_SWEEP, NUM_BINS_SWEEP, LR_SWEEP):
        where = {
            "AGENT_HYPERS.SIGMA_RATIO": sigma,
            "AGENT_HYPERS.NUM_BINS": bins,
            "AGENT_HYPERS.LR": lr,
        }
        runs = load_results(EXPERIMENT, name, where=where)
        if runs is not None:
            results[environment, support, sigma, bins, lr] = runs

In [ ]:
# Compute each config's lifetime scores and learning curve, and each environment's best config.
# A config is (support, SIGMA_RATIO, NUM_BINS, LR).
scores = {}
curves = {}
for (environment, *grid_point), runs in results.items():
    returns = return_curve(runs.reward, runs.done)
    scores.setdefault(environment, {})[tuple(grid_point)] = lifetime_average(returns)
    timesteps, values = subsample(returns)
    curves[environment, tuple(grid_point)] = (timesteps, mean_ci(values))
sensitivity = {
    environment: {point: mean_ci(runs) for point, runs in by_point.items()}
    for environment, by_point in scores.items()
}
best = {environment: best_by_mean(by_point) for environment, by_point in scores.items()}

def lr_power(lr):
    """An LR from the 4^-k grid as its exponent k, e.g. 0.000977 -> 5."""
    return round(-np.log(lr) / np.log(4))


def lr_tex(lr):
    """An LR as a mathtext power of four, for titles, legends and ticks."""
    return f"$4^{{-{lr_power(lr)}}}$"


TOP_K = 5
for environment, by_point in sensitivity.items():
    ranked = sorted(by_point.items(), key=lambda item: item[1][0], reverse=True)
    print(f"[{environment}] top {TOP_K} of {len(ranked)} configs (support, σ/bin, bins, LR):")
    for (support, sigma, bins, lr), (mean, low, high) in ranked[:TOP_K]:
        print(
            f"  ±{support}  σ/bin={sigma:<4}  bins={bins:<3}  LR=4^-{lr_power(lr)}  "
            f"{mean:8.1f}  [{low:.1f}, {high:.1f}]"
        )

In [ ]:
# Style for the grid-sweep plots.
STYLE = {
    "figure.figsize": (9, 6),
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
BAND_ALPHA = 0.2
TITLES = {"cartpole": "Cartpole", "mountaincar": "MountainCar"}
# Headroom past each task's best return, so curves and bands at the ceiling stay visible.
YLIMS = {"cartpole": (0, 525), "mountaincar": (-1000, 25)}
YLABEL_KW = {"rotation": 0, "ha": "right", "va": "center", "labelpad": 12}
SCORE_YLABEL = "Weighted\nlifetime\nreturn"
LR_XLABEL = "Learning rate"
SIGMA_XLABEL = "SIGMA_RATIO (σ / bin width)"
CURVE_XLABEL = "Timestep"
CURVE_YLABEL = "Return"
LEGEND = {"loc": "best"}
LR_TITLE = "{title}: LR sensitivity (±{support}, {bins} bins, 95% CI)"
SIGMA_TITLE = "{title}: ±{support}, LR={lr} (95% CI)"
CURVE_TITLE = "{title}: best config per σ/bin (95% CI)"
CURVE_LABEL = "σ/bin={sigma}, {bins} bins, ±{support}, LR={lr}"
PANEL_WIDTH = 9


def ordered_colors(values, cmap=plt.cm.viridis):
    """One colour per value, in order, so a swept hyper reads as a gradient."""
    return dict(zip(values, cmap(np.linspace(0, 0.9, len(values))), strict=True))


def draw_band(ax, x, mean, low, high, color, label, **line):
    ax.fill_between(x, low, high, color=color, alpha=BAND_ALPHA)
    ax.plot(x, mean, color=color, label=label, **line)

In [ ]:
# Plot LR sensitivity per environment: one line per SIGMA_RATIO at the best support and bins.
figures = {}
for environment in ENVIRONMENTS:
    if environment not in best:
        continue
    support, _, bins, _ = best[environment]
    colors = ordered_colors(SIGMA_RATIO_SWEEP)
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        for sigma in SIGMA_RATIO_SWEEP:
            points = [(support, sigma, bins, lr) for lr in LR_SWEEP]
            lrs = [p[3] for p in points if p in sensitivity[environment]]
            if not lrs:
                continue
            intervals = [sensitivity[environment][support, sigma, bins, lr] for lr in lrs]
            mean, low, high = np.array(intervals).T
            draw_band(ax, lrs, mean, low, high, colors[sigma], f"σ/bin={sigma}", marker="o")
        ax.set_xscale("log")
        ax.set_xticks(LR_SWEEP, [lr_tex(lr) for lr in LR_SWEEP])
        ax.minorticks_off()
        title = LR_TITLE.format(title=TITLES[environment], support=support, bins=bins)
        ax.set(title=title, xlabel=LR_XLABEL)
        ax.set_ylabel(SCORE_YLABEL, **YLABEL_KW)
        ax.legend(**LEGEND)
        fig.tight_layout()
    figures[f"{environment}_lr_sensitivity"] = fig
plt.show()

In [ ]:
# Plot SIGMA_RATIO sensitivity per environment: a panel per support, a line per NUM_BINS, at the best LR.
for environment in ENVIRONMENTS:
    if environment not in best:
        continue
    *_, lr = best[environment]
    colors = ordered_colors(NUM_BINS_SWEEP, plt.cm.plasma)
    with plt.rc_context(STYLE):
        _, height = STYLE["figure.figsize"]
        fig, axes = plt.subplots(
            1, len(SUPPORT_RANGES), figsize=(PANEL_WIDTH * len(SUPPORT_RANGES), height),
            sharey=True, squeeze=False,
        )
        for ax, support in zip(axes[0], SUPPORT_RANGES, strict=True):
            for bins in NUM_BINS_SWEEP:
                sigmas = [
                    s for s in SIGMA_RATIO_SWEEP
                    if (support, s, bins, lr) in sensitivity[environment]
                ]
                if not sigmas:
                    continue
                intervals = [sensitivity[environment][support, s, bins, lr] for s in sigmas]
                mean, low, high = np.array(intervals).T
                draw_band(ax, sigmas, mean, low, high, colors[bins], f"{bins} bins", marker="o")
            ax.set_xticks(SIGMA_RATIO_SWEEP)
            title = SIGMA_TITLE.format(title=TITLES[environment], support=support, lr=lr_tex(lr))
            ax.set(title=title, xlabel=SIGMA_XLABEL)
            ax.legend(**LEGEND)
        axes[0][0].set_ylabel(SCORE_YLABEL, **YLABEL_KW)
        fig.tight_layout()
    figures[f"{environment}_sigma_sensitivity"] = fig
plt.show()

In [ ]:
# Plot learning curves per environment: the best config for each SIGMA_RATIO.
for environment in ENVIRONMENTS:
    if environment not in scores:
        continue
    colors = ordered_colors(SIGMA_RATIO_SWEEP)
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        for sigma in SIGMA_RATIO_SWEEP:
            by_point = {p: s for p, s in scores[environment].items() if p[1] == sigma}
            if not by_point:
                continue
            point = best_by_mean(by_point)
            support, _, bins, lr = point
            timesteps, (mean, low, high) = curves[environment, point]
            label = CURVE_LABEL.format(sigma=sigma, bins=bins, support=support, lr=lr_tex(lr))
            width = 4 if point == best[environment] else STYLE["lines.linewidth"]
            draw_band(ax, timesteps, mean, low, high, colors[sigma], label, linewidth=width)
        ax.set(title=CURVE_TITLE.format(title=TITLES[environment]))
        ax.set(xlabel=CURVE_XLABEL, ylim=YLIMS[environment])
        ax.set_ylabel(CURVE_YLABEL, **YLABEL_KW)
        ax.set_xticks([0, timesteps[-1]])  # timestep axis: show only 0 and the max
        ax.legend(**LEGEND)
        fig.tight_layout()
    figures[f"{environment}_learning_curves"] = fig
plt.show()

In [ ]:
# Save every plotted figure as a PDF and a PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(PLOTS_DIR / f"{name}.png", bbox_inches="tight", dpi=200)
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")